## ESPnet TTS Demonstracija

### 0. Instaliuojame bibliotekas

In [ ]:
# !apt-get update
!apt-get install -y espeak-ng

!pip install "espnet[tts] @ git+https://github.com/airenas/espnet.git@demo.v01" phonemizer -q --disable-pip-version-chec
!pip install parallel_wavegan -q --disable-pip-version-check

print("\nDONE: paruošta")

### 0.1 Pataisome PWGan dėl Python 3.12

In [ ]:
## workaround: fix parallel_wavegan for python3.12
import scipy.signal.windows
import sys
sys.modules['scipy.signal'].kaiser = scipy.signal.windows.kaiser

### 1. Inicializuojame ESPnet ir pagalbines funkcijas

In [ ]:
from espnet2.bin.tts_inference import Text2Speech
import soundfile as sf
from IPython.display import Audio, HTML
from huggingface_hub import snapshot_download
from pathlib import Path

device = "cpu"
# device = "cuda" # uncomment if you have a GPU and want to use it

def download_vocoder(tag: str) -> str:
    repo_dir = snapshot_download(tag)
    repo_path = Path(repo_dir)

    for file in repo_path.rglob("*.pkl"):
        # print(f"found {str(file)}")
        return str(file)

    raise FileNotFoundError("No *.pkl file found in the vocoder repo")
   
class TTSData:
    def __init__(self, tts, info):
        self.tts = tts
        self.info = info


def init_tts(am_tag=None, vocoder_tag=None, info=""):
    voc_str = vocoder_tag
    if not voc_str:
        voc_str = "Griffin-Lim"
    
    print(f"\n===================================\nAM      = {am_tag}")
    print(f"Vocoder = {voc_str}")
    print(f"===================================")

    vocoder_file=None
    if vocoder_tag:
        vocoder_file = download_vocoder(vocoder_tag)
        vocoder_tag = None
    
    tts = Text2Speech.from_pretrained(
        model_tag=am_tag,
        vocoder_tag=vocoder_tag,
        vocoder_file=vocoder_file,
        device=device
    )
    return TTSData(tts, info)

print ("\nDONE: ESPnet paruoštas")    

### 2. Parsiunčiame/užkrauname modelius iš HuggingFace

Paruošti tokie modeliai:
1. AM vyriškas balsas: [VSSA-SDSA/sing-arn.fastspeech2.v01](https://huggingface.co/VSSA-SDSA/sing-arn.fastspeech2.v01)
2. AM moteriškas balsas - [VSSA-SDSA/sing-agn.fastspeech2.v01](https://huggingface.co/VSSA-SDSA/sing-agn.fastspeech2.v01)
4. Vokoderis vyriškas balsas - [VSSA-SDSA/sing-arn.vocoder.style_melgan.v01](https://huggingface.co/VSSA-SDSA/sing-arn.vocoder.style_melgan.v01)
5. Vokoderis moteriškas balsas - [VSSA-SDSA/sing-agn.vocoder.style_melgan.v01](https://huggingface.co/VSSA-SDSA/sing-agn.vocoder.style_melgan.v01)


In [ ]:
am_tag_hf_arn="VSSA-SDSA/sing-arn.fastspeech2.v01"
vocoder_tag_hf_arn= "VSSA-SDSA/sing-arn.vocoder.style_melgan.v01"
am_tag_hf_agn="VSSA-SDSA/sing-agn.fastspeech2.v01"
vocoder_tag_hf_agn= "VSSA-SDSA/sing-agn.vocoder.style_melgan.v01"

tts_hf_arn = init_tts(am_tag=am_tag_hf_arn, vocoder_tag=vocoder_tag_hf_arn, info="arn-fastspeech2+arn-style")
tts_hf_agn = init_tts(am_tag=am_tag_hf_agn, vocoder_tag=vocoder_tag_hf_agn, info="agn-fastspeech2+agn-style")

tts_list = [tts_hf_arn, tts_hf_agn]

print (f"\nDONE: {len(tts_list)} modeliai paruošti\n")  

### 3. Rezultatas

Sintezuojamas tekstas

In [ ]:
texts = ["Sveiki, aš naujas lietuviškas balsas.", 
         "Aš esu labai gerai įrašytas.", 
         "Mus pasivijo juodas prabangus automobilis.",
         "Ji artinasi prie uolos, plačia krūtine skrosdama bangas, tarsi laivas, kuris lyg sparnais skrieja vilnimis, irkluojamas galingų vyrų."
        ]
for i, text in enumerate(texts):
    print(f"\n==========================================================================\nText = {text}")
    for it, tts_data in enumerate(tts_list):
        tts = tts_data.tts
        wav = tts(text)["wav"]
        filename = f"output_{i}_{it}.wav"
        sf.write(filename, wav.cpu().numpy(), tts.fs)
        audio_widget = Audio(filename)._repr_html_()
        display(HTML(f"<div style='display:flex; align-items:center; gap:10px;'>{audio_widget}<span>{tts_data.info}:</span></div>"))
